# WSJ 1965-2014 → Sampled Financial Articles (TDM Studio)

Schema (confirmed from sample 134001711.xml):
- Date: `<NumericDate>1975-01-09</NumericDate>` (also `<StartDate>`)
- Title: `<TitleAtt><Title>...</Title></TitleAtt>`
- Body: `<TextInfo><HiddenText HTMLContent="true">&lt;html&gt;...&lt;p&gt;...&lt;/p&gt;...&lt;/html&gt;</HiddenText></TextInfo>` — HTML-escaped HTML inside XML, needs double-decoding

Pipeline (4 cells):
1. **Index** — fast regex over every XML for date + financial-keyword flag (~few minutes)
2. **Sample** — pick N/month from financial subset
3. **Parse + save** — full XML parse on only sampled files; HTML-decode body; write parquet
4. **Export** — `aws s3 cp` to your results bucket

In [ ]:
# ===== Cell 1: Build date index from all XML files =====
import os, re, time
from pathlib import Path
import pandas as pd

candidates = [Path('data/WSJ_1965-2014'), Path('/data/WSJ_1965-2014'),
              Path('../data/WSJ_1965-2014'), Path('/home/jovyan/data/WSJ_1965-2014')]
WSJ_ROOT = next((p for p in candidates if p.exists()), None)
if WSJ_ROOT is None:
    raise SystemExit('No WSJ folder found at any candidate path.')
print(f'WSJ_ROOT = {WSJ_ROOT}')

OUT_DIR    = Path('../ProQuest TDM Studio Samples/output_files/')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / 'wsj_index.csv'

# Date in <NumericDate> per confirmed sample; <StartDate> as backup
DATE_RE = re.compile(rb'<NumericDate>\s*(\d{4}-\d{2}-\d{2})', re.I)
DATE_RE_BACKUP = re.compile(rb'<StartDate>\s*(\d{4}-?\d{2}-?\d{2})', re.I)

FIN_KW = re.compile(
    rb'stock\s*market|stock\s*exchange|wall\s*street|'
    rb'\bdow\b|\bs\s*&\s*p\s*500\b|\bnasdaq\b|'
    rb'\bequit(y|ies)\b|\bsecurit(y|ies)\b|\bbond[s]?\b|\bdividend[s]?\b|'
    rb'\bearnings?\b|\bprofit[s]?\b|\brevenue[s]?\b|'
    rb'\bcrash(ed)?\b|\bbull(ish)?\b|\bbear(ish)?\b|'
    rb'\bfederal\s*reserve\b|\bfomc\b|\bdiscount\s*rate\b|'
    rb'\binflation\b|\bdeflation\b|\brecession\b|\bunemploy',
    re.I,
)

def extract_date(buf):
    m = DATE_RE.search(buf) or DATE_RE_BACKUP.search(buf)
    return m.group(1).decode() if m else None

files = list(WSJ_ROOT.glob('*.xml'))
print(f'Total XML files: {len(files):,}')

rows = []
t0 = time.time()
for i, f in enumerate(files):
    try:
        buf = f.read_bytes()
    except Exception:
        continue
    d = extract_date(buf)
    if d is None:
        continue
    is_fin = bool(FIN_KW.search(buf))
    rows.append({'fname': f.name, 'date_raw': d, 'is_fin': is_fin})
    if (i + 1) % 5000 == 0:
        rate = (i + 1) / (time.time() - t0)
        eta = (len(files) - (i + 1)) / rate / 60
        print(f'  {i+1:,}/{len(files):,}  rate={rate:.0f}/s  ETA={eta:.1f} min', flush=True)

idx = pd.DataFrame(rows)
idx['date'] = pd.to_datetime(idx['date_raw'], errors='coerce')
idx = idx.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
idx['year']  = idx['date'].dt.year
idx['month'] = idx['date'].dt.month
idx.to_csv(INDEX_PATH, index=False)
print(f'\nIndexed {len(idx):,} dated files; {idx["is_fin"].sum():,} flagged financial.')
print(f'Date range: {idx["date"].min()} -> {idx["date"].max()}')
print(f'Articles per decade:')
print((idx.groupby((idx["year"]//10)*10).size()).to_string())
print(f'Saved index to {INDEX_PATH}')

In [ ]:
# ===== Cell 2: Sample N financial articles per month =====
import pandas as pd
from pathlib import Path

OUT_DIR    = Path('../ProQuest TDM Studio Samples/output_files/')
INDEX_PATH = OUT_DIR / 'wsj_index.csv'

SAMPLE_PER_MONTH = 10
YEAR_START = 1965
YEAR_END   = 2014

idx = pd.read_csv(INDEX_PATH, parse_dates=['date'])
fin = idx[(idx['is_fin']) & (idx['year'] >= YEAR_START) & (idx['year'] <= YEAR_END)].copy()
print(f'Financial-flagged in {YEAR_START}-{YEAR_END}: {len(fin):,}')

sampled = (fin.groupby(['year','month'], group_keys=False)
              .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_MONTH), random_state=0)))
sampled = sampled.sort_values('date').reset_index(drop=True)
print(f'Sampled {len(sampled):,} articles ({sampled["year"].nunique()} years, '
      f'{sampled.groupby(["year","month"]).ngroups} year-months)')
sampled.to_csv(OUT_DIR / 'wsj_sampled_index.csv', index=False)

In [ ]:
# ===== Cell 3: Parse sampled XMLs (date, title, body) → parquet =====
import os, re, html
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

candidates = [Path('data/WSJ_1965-2014'), Path('/data/WSJ_1965-2014'),
              Path('../data/WSJ_1965-2014'), Path('/home/jovyan/data/WSJ_1965-2014')]
WSJ_ROOT = next(p for p in candidates if p.exists())
OUT_DIR  = Path('../ProQuest TDM Studio Samples/output_files/')
OUT_PATH = OUT_DIR / 'wsj_1965_2014_financial.parquet'

MAX_BODY_CHARS = 1500
MIN_BODY_CHARS = 200

TAG_RE = re.compile(r'<[^>]+>')   # strip HTML tags after unescape

def html_to_text(s):
    """<HiddenText> contains HTML-escaped HTML. Decode then strip tags."""
    if not s: return ''
    decoded = html.unescape(s)              # &lt;p&gt; → <p>
    decoded = html.unescape(decoded)        # in case of double-escape
    plain = TAG_RE.sub(' ', decoded)        # strip <p>, <body>, etc.
    plain = re.sub(r'\s+', ' ', plain).strip()
    return plain

sampled = pd.read_csv(OUT_DIR / 'wsj_sampled_index.csv', parse_dates=['date'])

rows = []
for i, r in sampled.iterrows():
    fpath = WSJ_ROOT / r['fname']
    try:
        tree = ET.parse(fpath)
        root = tree.getroot()
    except Exception:
        continue
    # Headline: <TitleAtt><Title>...</Title></TitleAtt>
    title_node = root.find('.//Title')
    headline = (title_node.text or '').strip() if title_node is not None else ''
    # Body: <TextInfo><HiddenText>...</HiddenText></TextInfo>, with HTML-escaped HTML
    body_node = root.find('.//HiddenText')
    raw = body_node.text if body_node is not None else ''
    body = html_to_text(raw)[:MAX_BODY_CHARS]
    headline = re.sub(r'\s+', ' ', headline)[:200]
    if len(body) < MIN_BODY_CHARS:
        continue
    rows.append({
        'date': r['date'], 'year': r['year'], 'month': r['month'],
        'newspaper': 'Wall Street Journal', 'state': None, 'page': None,
        'headline': headline, 'text': body, 'text_len': len(body),
        'article_id': r['fname'],
    })
    if (i + 1) % 500 == 0:
        print(f'  parsed {i+1}/{len(sampled)}', flush=True)

df = pd.DataFrame(rows)[['date','year','month','newspaper','state','page','headline','text','text_len','article_id']]
df.to_parquet(OUT_PATH, index=False, compression='snappy')
size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f'\nWrote {len(df):,} articles to {OUT_PATH}')
print(f'File size: {size_mb:.2f} MB  (TDM export cap: 30 MB/week)')
if size_mb > 28:
    print('  WARNING close to cap — drop SAMPLE_PER_MONTH or MAX_BODY_CHARS.')
print(f'\nSample rows:')
print(df[['date','headline','text_len']].head(5).to_string(index=False))

In [ ]:
# ===== Cell 4: Ship parquet to your S3 results bucket =====
data_to_export = '../ProQuest TDM Studio Samples/output_files/wsj_1965_2014_financial.parquet'
!aws s3 cp "$data_to_export" s3://pq-tdm-studio-results/tdm-ale-data/a4992/results/